# [17.1] Checkpoint Archaeology and Mechanism Emergence - Solutions

This notebook runs the reference implementation, visible tests, live CPU checkpoint smoke path, and committed CUDA report checks.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter17_training_dynamics"
section = "part1_checkpoint_archaeology"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_checkpoint_archaeology.tests as tests
from chapter17_training_dynamics.exercises.part1_checkpoint_archaeology import solutions

## Visible Tests

In [ ]:
tests.test_first_threshold_crossing_finds_first_crossing_and_validates_inputs(
    solutions.first_threshold_crossing
)
tests.test_stable_threshold_step_requires_consecutive_checkpoints(
    solutions.stable_threshold_step
)
tests.test_mechanism_emergence_report_tracks_peak_and_monotonicity(
    solutions.mechanism_emergence_report
)
tests.test_phase_transition_report_detects_largest_adjacent_jump(
    solutions.phase_transition_report
)
tests.test_random_control_report_rejects_overstrong_control(
    solutions.random_control_report
)
tests.test_developmental_comparison_excludes_random_control_from_ordering(
    solutions.developmental_comparison_report
)
tests.test_checkpoint_emergence_smoke_test(solutions.checkpoint_emergence_smoke_test)
tests.test_phase_transition_smoke_test(solutions.phase_transition_smoke_test)
tests.test_random_control_smoke_test(solutions.random_control_smoke_test)
tests.test_developmental_comparison_smoke_test(solutions.developmental_comparison_smoke_test)
tests.test_live_checkpoint_archaeology_smoke_test_trains_saves_reloads_and_controls(
    solutions.live_checkpoint_archaeology_smoke_test
)
tests.test_notebook_contract(solutions.run_smoke_test)

## Notebook Contract

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["checkpoint_emergence"]["first_crossing_step"] == 300, "Toy AR first crossing should be step 300."
assert contract["checkpoint_emergence"]["stable_from_step"] == 300, "Toy AR stable emergence should start at step 300."
assert contract["phase_transition"]["transition_step"] == 300, "Largest toy AR jump should land at step 300."
assert contract["random_control"]["control_passed"], "Toy random-control trajectory should stay below threshold."
assert contract["developmental_comparison"]["earliest_family"] == "jepa", "Toy JEPA trajectory should emerge earliest."
assert contract["developmental_comparison"]["latest_family"] == "diffusion", "Toy diffusion trajectory should emerge latest."
live = contract["live_checkpoint_archaeology"]
assert live["preflight_passed"], "Live checkpoint smoke test should pass."
assert live["real_checkpoints_reloaded"], "Live metrics should come from reloaded checkpoints."
assert live["complete_finite_domain_evaluated"] and not live["ood_generalization_claimed"], "Mod-13 scope should be finite-domain, not OOD."
assert live["checkpoint_count"] == 26, "Live smoke should write target and random-control checkpoints."
contract

## Real CUDA Report Evidence

In [ ]:
tests.test_committed_gpu_report_records_real_checkpoint_preflight()
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"], "The committed verification report should be accepted."
assert gpu["cuda_available"], "The 17.1 release report should be a CUDA report."
assert gpu["preflight_passed"], "The real checkpoint archaeology preflight should pass."
assert gpu["real_checkpoints_reloaded"], "Metrics should be computed from reloaded checkpoint files."
assert gpu["model_family"] == "tiny_modular_addition_mlp", "Unexpected model family in report."
assert gpu["modulus"] == 13, "The finite model organism should use mod-13 addition."
assert gpu["table_example_count"] == 169, "The complete mod-13 addition table has 169 examples."
assert gpu["complete_finite_domain_evaluated"], "The report should evaluate the complete finite mod-13 domain."
assert not gpu["ood_generalization_claimed"], "The report should not claim OOD generalization."
assert gpu["checkpoint_count"] == 26, "Target and random-control runs should each save 13 checkpoints."
assert gpu["stable_from_step"] == 30, "Stable emergence should begin at step 30."
assert gpu["final_accuracy"] == 1.0, "The target run should reach exact table accuracy."
assert gpu["random_control_passed"], "The random-label checkpoint control should be rejected as non-emergent."
assert gpu["random_control_peak_accuracy"] <= 0.2, "The random-label control should remain near chance."
assert gpu["within_vram_budget"], "The preflight should remain inside the declared VRAM budget."

{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "real_checkpoints_reloaded",
    "complete_finite_domain_evaluated",
    "ood_generalization_claimed",
    "stable_from_step",
    "final_accuracy",
    "random_control_peak_accuracy",
    "phase_transition_jump",
    "peak_vram_gb",
]}